# ⚡ Notebook 02 — Event Stream Simulator
## Projeto: Otimização de Frotas Citi Bike NYC
### Integrante 1 — Streaming & Event Simulator

---

**Objetivo:** Simular um fluxo de dados em tempo real de viagens do Citi Bike, reproduzindo a "pressão logística" que o sistema sofre ao longo do dia. O simulador lê dados históricos da camada Bronze e os emite como micro-lotes (micro-batches), simulando um cenário de streaming.

**Arquitetura:**
```
┌─────────────────────┐     ┌─────────────────────┐     ┌──────────────────┐
│  Bronze/trips       │ ──► │  Event Simulator     │ ──► │  streaming/       │
│  (Parquet histórico)│     │  (Python Generator)  │     │  events/          │
└─────────────────────┘     │  - Micro-batches     │     │  (JSON micro-lotes│
                            │  - Velocidade ajust. │     │   particionados)  │
┌─────────────────────┐     │  - Enriquecimento    │     └──────────────────┘
│  GBFS Station Feed  │ ──► │    com station status│              │
│  (tempo real)       │     └─────────────────────┘              ▼
└─────────────────────┘                              ┌──────────────────┐
                                                     │  Spark Structured│
                                                     │  Streaming       │
                                                     │  (consumidor)    │
                                                     └──────────────────┘
```

**Ferramentas de Big Data:**
- **Spark Structured Streaming** — Consumidor dos micro-lotes
- **PySpark** — Processamento do stream
- **JSON micro-batch** — Formato intermediário do stream

## 0. Setup

In [1]:
!pip install pyspark pyarrow -q

In [2]:
import os
import sys
import json
import time
import shutil
import random
import logging
import threading
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, IntegerType, LongType
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('citibike_streaming')


In [3]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================

PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / 'dados'
BRONZE_TRIPS = DATA_DIR / 'bronze' / 'trips'
BRONZE_STATIONS = DATA_DIR / 'bronze' / 'stations'

# Diretórios do streaming — usar filesystem Linux (evita problemas de permissão no WSL)
STREAM_DIR = Path.home() / 'citibike' / 'streaming'
STREAM_EVENTS = STREAM_DIR / 'events'
STREAM_CHECKPOINT = STREAM_DIR / 'checkpoint'
STREAM_OUTPUT = STREAM_DIR / 'output'

# Limpar diretórios de streaming (fresh start)
for d in [STREAM_EVENTS, STREAM_CHECKPOINT, STREAM_OUTPUT]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

# Parâmetros do simulador
BATCH_SIZE = 50
BATCH_INTERVAL_SEC = 2.0
TOTAL_BATCHES = 100
SIMULATION_DATE = '2024-07-15'
SPEED_FACTOR = 60

print(f"📁 Stream events dir: {STREAM_EVENTS}")
print(f"⚙️  Batch size: {BATCH_SIZE} viagens")
print(f"⏱️  Intervalo: {BATCH_INTERVAL_SEC}s")
print(f"📅  Data simulada: {SIMULATION_DATE}")
print(f"⚡ Speed factor: {SPEED_FACTOR}x")

📁 Stream events dir: /home/joao/citibike/streaming/events
⚙️  Batch size: 50 viagens
⏱️  Intervalo: 2.0s
📅  Data simulada: 2024-07-15
⚡ Speed factor: 60x


In [4]:
# Inicializar Spark
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("CitiBike_StreamSimulator")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "4")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark {spark.version} inicializado")
print(f"   Cores: {spark.sparkContext.defaultParallelism}")
print(f"   Driver memory: {spark.conf.get('spark.driver.memory')}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/29 23:44:04 WARN Utils: Your hostname, DESKTOP-K9JUL83, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/29 23:44:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 23:44:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 4.1.1 inicializado
   Cores: 12
   Driver memory: 4g


---
## 1. 🏗️ Preparar Dados para Simulação

Carregamos um dia de dados históricos da Bronze e os ordenamos cronologicamente para reproduzir o fluxo real.

In [5]:
# ============================================================
# 1.1 Carregar dados de um dia da Bronze
# ============================================================

sim_date = pd.Timestamp(SIMULATION_DATE)

partition_path = BRONZE_TRIPS / f"year={sim_date.year}" / f"month={sim_date.month}"

if partition_path.exists() and any(partition_path.rglob("*.parquet")):
    logger.info(f"📖 Lendo dados de {partition_path}...")
    day_df = (
        spark.read.parquet(str(partition_path))
        .filter(F.to_date("started_at") == SIMULATION_DATE)
        .orderBy("started_at")
    )
    total_trips = day_df.count()
    logger.info(f"📊 Viagens no dia {SIMULATION_DATE}: {total_trips:,}")
    day_df.show(5, truncate=False)
else:
    logger.warning(f"⚠️  Partição não encontrada em {partition_path}")
    logger.info("🔄 Gerando dados sintéticos para demonstração...")
    total_trips = 0


23:44:07 [INFO] 📖 Lendo dados de /mnt/c/Users/PC Gamer/Documents/teste/dados/bronze/trips/year=2024/month=7...
23:44:10 [INFO] 📊 Viagens no dia 2024-07-15: 146,831                           
[Stage 4:===================================>                      (8 + 5) / 13]

+----------------+-------------+-----------------------+-----------------------+---------------------------+----------------+---------------------------+--------------+-----------------+------------------+-----------+------------+-------------+-----------------+--------------------------+
|ride_id         |rideable_type|started_at             |ended_at               |start_station_name         |start_station_id|end_station_name           |end_station_id|start_lat        |start_lng         |end_lat    |end_lng     |member_casual|trip_duration_sec|ingestion_timestamp       |
+----------------+-------------+-----------------------+-----------------------+---------------------------+----------------+---------------------------+--------------+-----------------+------------------+-----------+------------+-------------+-----------------+--------------------------+
|A97DF22F69121B25|classic_bike |2024-07-15 00:00:00.029|2024-07-15 00:04:44.557|Crown St & Troy Ave        |3751.05         |Easte

In [6]:
# ============================================================
# 1.2 Fallback: Gerador de dados sintéticos
# ============================================================

# Estações mais populares do Citi Bike (dados reais)
POPULAR_STATIONS = [
    {'id': '6681.01', 'name': 'Broadway & W 60 St', 'lat': 40.7691, 'lng': -73.9819, 'capacity': 55},
    {'id': '6140.06', 'name': 'Lincoln Center', 'lat': 40.7725, 'lng': -73.9832, 'capacity': 39},
    {'id': '5905.09', 'name': 'University Pl & E 14 St', 'lat': 40.7340, 'lng': -73.9920, 'capacity': 51},
    {'id': '6584.14', 'name': 'Grand Army Plaza & Central Park S', 'lat': 40.7644, 'lng': -73.9732, 'capacity': 63},
    {'id': '5788.03', 'name': 'W 21 St & 6 Ave', 'lat': 40.7416, 'lng': -73.9943, 'capacity': 43},
    {'id': '6494.05', 'name': 'E 53 St & Madison Ave', 'lat': 40.7594, 'lng': -73.9739, 'capacity': 35},
    {'id': '5491.09', 'name': 'Central Park W & W 72 St', 'lat': 40.7757, 'lng': -73.9762, 'capacity': 47},
    {'id': '5901.09', 'name': 'Bergen St & Smith St', 'lat': 40.6862, 'lng': -73.9901, 'capacity': 25},
    {'id': '5267.08', 'name': 'Fulton St & Broadway', 'lat': 40.7101, 'lng': -74.0095, 'capacity': 41},
    {'id': '5493.02', 'name': 'Pier 40 - Hudson River Park', 'lat': 40.7277, 'lng': -74.0116, 'capacity': 59},
    {'id': '6710.04', 'name': '1 Ave & E 68 St', 'lat': 40.7652, 'lng': -73.9584, 'capacity': 30},
    {'id': '6254.11', 'name': 'Broadway & E 22 St', 'lat': 40.7403, 'lng': -73.9892, 'capacity': 39},
    {'id': '5347.03', 'name': 'Clark St & Henry St', 'lat': 40.6976, 'lng': -73.9930, 'capacity': 24},
    {'id': '7654.01', 'name': 'Bedford Ave & N 7 St', 'lat': 40.7183, 'lng': -73.9572, 'capacity': 27},
    {'id': '6116.02', 'name': 'W 41 St & 8 Ave', 'lat': 40.7565, 'lng': -73.9904, 'capacity': 45},
]

def demand_profile(hour):
    """
    Retorna um multiplicador de demanda baseado na hora do dia.
    Simula os picos de manhã (commute) e tarde (volta + lazer).
    """
    profiles = {
        0: 0.1, 1: 0.05, 2: 0.03, 3: 0.02, 4: 0.03, 5: 0.08,
        6: 0.25, 7: 0.65, 8: 1.0, 9: 0.85, 10: 0.55, 11: 0.60,
        12: 0.70, 13: 0.65, 14: 0.60, 15: 0.65, 16: 0.80, 17: 1.0,
        18: 0.95, 19: 0.75, 20: 0.55, 21: 0.40, 22: 0.25, 23: 0.15
    }
    return profiles.get(hour, 0.3)

def generate_synthetic_trip(sim_time):
    """Gera uma viagem sintética realista."""
    start_station = random.choice(POPULAR_STATIONS)
    end_station = random.choice([s for s in POPULAR_STATIONS if s['id'] != start_station['id']])
    
    # Duração baseada na distância aproximada
    dist = ((start_station['lat']-end_station['lat'])**2 + 
            (start_station['lng']-end_station['lng'])**2) ** 0.5
    duration_sec = int(max(120, min(3600, dist * 100000 + random.gauss(0, 120))))
    
    bike_type = random.choices(
        ['classic_bike', 'electric_bike'],
        weights=[0.55, 0.45]
    )[0]
    
    member_type = random.choices(
        ['member', 'casual'],
        weights=[0.70, 0.30]
    )[0]
    
    end_time = sim_time + timedelta(seconds=duration_sec)
    
    return {
        'ride_id': f"SIM_{sim_time.strftime('%Y%m%d%H%M%S')}_{random.randint(1000,9999)}",
        'rideable_type': bike_type,
        'started_at': sim_time.isoformat(),
        'ended_at': end_time.isoformat(),
        'start_station_name': start_station['name'],
        'start_station_id': start_station['id'],
        'end_station_name': end_station['name'],
        'end_station_id': end_station['id'],
        'start_lat': start_station['lat'] + random.gauss(0, 0.0001),
        'start_lng': start_station['lng'] + random.gauss(0, 0.0001),
        'end_lat': end_station['lat'] + random.gauss(0, 0.0001),
        'end_lng': end_station['lng'] + random.gauss(0, 0.0001),
        'member_casual': member_type,
        'trip_duration_sec': duration_sec,
    }

print("🔧 Gerador de dados sintéticos pronto")
print(f"   Estações populares carregadas: {len(POPULAR_STATIONS)}")
print(f"   Exemplo de viagem:")
sample = generate_synthetic_trip(datetime(2024, 7, 15, 8, 30, 0))
for k, v in sample.items():
    print(f"     {k}: {v}")

🔧 Gerador de dados sintéticos pronto
   Estações populares carregadas: 15
   Exemplo de viagem:
     ride_id: SIM_20240715083000_1743
     rideable_type: classic_bike
     started_at: 2024-07-15T08:30:00
     ended_at: 2024-07-15T09:30:00
     start_station_name: Clark St & Henry St
     start_station_id: 5347.03
     end_station_name: Bedford Ave & N 7 St
     end_station_id: 7654.01
     start_lat: 40.697567596948765
     start_lng: -73.99294778923608
     end_lat: 40.718220221197
     end_lng: -73.95701761709977
     member_casual: member
     trip_duration_sec: 3600


---
## 2. ⚡ Event Stream Simulator (Produtor)

O simulador funciona assim:
1. Lê dados do dia da Bronze (ou gera sintéticos)
2. Ordena por `started_at`
3. Emite micro-lotes de N viagens como arquivos JSON no diretório `streaming/events/`
4. Cada arquivo é um micro-lote que o Spark Structured Streaming detecta automaticamente

Isso simula um cenário real onde eventos de viagem chegam continuamente.

In [7]:
# ============================================================
# 2.1 Classe principal do simulador
# ============================================================

class CitiBikeEventSimulator:
    """
    Simulador de stream de eventos de viagem do Citi Bike.
    
    Pode operar em dois modos:
    - 'historical': lê dados reais da Bronze e os emite cronologicamente
    - 'synthetic': gera dados sintéticos com perfil de demanda realista
    
    Os eventos são escritos como JSON micro-batches no diretório de output,
    que pode ser lido pelo Spark Structured Streaming.
    """
    
    def __init__(self, output_dir, batch_size=50, interval_sec=2.0, speed_factor=60):
        self.output_dir = Path(output_dir)
        self.batch_size = batch_size
        self.interval_sec = interval_sec
        self.speed_factor = speed_factor
        self.batch_count = 0
        self.total_events = 0
        self.running = False
        self.station_balance = defaultdict(lambda: {'bikes': 10, 'docks': 20})
        
        # Inicializar balanço das estações
        for s in POPULAR_STATIONS:
            cap = s['capacity']
            bikes = int(cap * 0.6)  # 60% cheio inicialmente
            self.station_balance[s['id']] = {'bikes': bikes, 'docks': cap - bikes}
        
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def _update_station_balance(self, event):
        """Atualiza o balanço simulado de bikes nas estações."""
        start_id = event['start_station_id']
        end_id = event['end_station_id']
        
        # Bike sai da estação de origem
        if self.station_balance[start_id]['bikes'] > 0:
            self.station_balance[start_id]['bikes'] -= 1
            self.station_balance[start_id]['docks'] += 1
        
        # Bike chega na estação de destino
        if self.station_balance[end_id]['docks'] > 0:
            self.station_balance[end_id]['bikes'] += 1
            self.station_balance[end_id]['docks'] -= 1
    
    def _compute_pressure_metrics(self, events_batch):
        """
        Calcula métricas de "pressão logística" para o micro-lote:
        - Estações com deficit de bikes
        - Estações com deficit de docks
        - Índice de desbalanceamento
        """
        low_bikes = sum(1 for s in self.station_balance.values() if s['bikes'] <= 2)
        low_docks = sum(1 for s in self.station_balance.values() if s['docks'] <= 2)
        total_stations = len(self.station_balance)
        
        # Índice de pressão: 0 = balanceado, 1 = totalmente desbalanceado
        pressure_index = (low_bikes + low_docks) / (2 * max(total_stations, 1))
        
        return {
            'stations_low_bikes': low_bikes,
            'stations_low_docks': low_docks,
            'pressure_index': round(pressure_index, 4),
            'total_active_stations': total_stations,
        }
    
    def emit_batch_synthetic(self, sim_time):
        """Emite um micro-lote de eventos sintéticos."""
        hour = sim_time.hour
        demand = demand_profile(hour)
        actual_batch_size = max(1, int(self.batch_size * demand))
        
        events = []
        for i in range(actual_batch_size):
            # Adicionar variação temporal dentro do lote
            offset_sec = random.randint(0, int(self.interval_sec * self.speed_factor))
            event_time = sim_time + timedelta(seconds=offset_sec)
            
            event = generate_synthetic_trip(event_time)
            self._update_station_balance(event)
            events.append(event)
        
        # Adicionar metadata do lote
        pressure = self._compute_pressure_metrics(events)
        
        batch_data = {
            'batch_id': self.batch_count,
            'batch_timestamp': datetime.now().isoformat(),
            'simulation_time': sim_time.isoformat(),
            'num_events': len(events),
            'pressure_metrics': pressure,
            'events': events,
        }
        
        return batch_data, events
    
    def emit_batch_historical(self, trips_chunk):
        """Emite um micro-lote a partir de dados históricos reais."""
        events = []
        for row in trips_chunk:
            event = {
                'ride_id': row['ride_id'],
                'rideable_type': row['rideable_type'],
                'started_at': str(row['started_at']),
                'ended_at': str(row['ended_at']),
                'start_station_name': row['start_station_name'],
                'start_station_id': str(row['start_station_id']),
                'end_station_name': row['end_station_name'],
                'end_station_id': str(row['end_station_id']),
                'start_lat': float(row['start_lat']) if row['start_lat'] else None,
                'start_lng': float(row['start_lng']) if row['start_lng'] else None,
                'end_lat': float(row['end_lat']) if row['end_lat'] else None,
                'end_lng': float(row['end_lng']) if row['end_lng'] else None,
                'member_casual': row['member_casual'],
                'trip_duration_sec': int(row['trip_duration_sec']) if row['trip_duration_sec'] else None,
            }
            self._update_station_balance(event)
            events.append(event)
        
        pressure = self._compute_pressure_metrics(events)
        
        batch_data = {
            'batch_id': self.batch_count,
            'batch_timestamp': datetime.now().isoformat(),
            'simulation_time': events[0]['started_at'] if events else None,
            'num_events': len(events),
            'pressure_metrics': pressure,
            'events': events,
        }
        
        return batch_data, events
    
    def write_batch(self, batch_data):
        """Escreve o micro-lote como arquivo JSON."""
        filename = f"batch_{self.batch_count:06d}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.json"
        filepath = self.output_dir / filename
        
        # Extrair apenas os eventos para o arquivo (formato que o Spark lê)
        # Cada linha é um JSON (formato JSON Lines para Spark)
        with open(filepath, 'w') as f:
            for event in batch_data['events']:
                f.write(json.dumps(event) + '\n')
        
        self.batch_count += 1
        self.total_events += batch_data['num_events']
        
        return filepath
    
    def run_synthetic(self, total_batches=100, simulation_date='2024-07-15'):
        """
        Executa o simulador no modo sintético.
        Simula um dia inteiro de operação acelerado.
        """
        self.running = True
        sim_time = datetime.fromisoformat(f"{simulation_date}T05:00:00")
        end_time = datetime.fromisoformat(f"{simulation_date}T23:59:59")
        
        logger.info(f"🚀 Simulador iniciado (modo SINTÉTICO)")
        logger.info(f"   Simulando: {sim_time} → {end_time}")
        logger.info(f"   Speed factor: {self.speed_factor}x")
        logger.info(f"   Batches planejados: {total_batches}")
        
        while self.running and self.batch_count < total_batches and sim_time < end_time:
            batch_data, events = self.emit_batch_synthetic(sim_time)
            filepath = self.write_batch(batch_data)
            
            pressure = batch_data['pressure_metrics']
            
            if self.batch_count % 10 == 0 or self.batch_count <= 3:
                logger.info(
                    f"📤 Batch {self.batch_count:4d} | "
                    f"SimTime: {sim_time.strftime('%H:%M')} | "
                    f"Events: {len(events):3d} | "
                    f"Pressure: {pressure['pressure_index']:.2f} | "
                    f"Low bikes: {pressure['stations_low_bikes']} | "
                    f"Total: {self.total_events:,}"
                )
            
            # Avançar tempo simulado
            sim_time += timedelta(seconds=self.interval_sec * self.speed_factor)
            time.sleep(self.interval_sec)
        
        self.running = False
        logger.info(f"\n🏁 Simulação encerrada!")
        logger.info(f"   Batches emitidos: {self.batch_count}")
        logger.info(f"   Total de eventos: {self.total_events:,}")
    
    def run_historical(self, spark_df, total_batches=0):
        """
        Executa o simulador usando dados históricos reais da Bronze.
        """
        self.running = True
        
        # Converter para Pandas para iterar (dados de 1 dia cabem na memória)
        logger.info("📖 Convertendo dados históricos para iteração...")
        trips_pd = spark_df.toPandas()
        trips_list = trips_pd.to_dict('records')
        
        logger.info(f"🚀 Simulador iniciado (modo HISTÓRICO)")
        logger.info(f"   Viagens disponíveis: {len(trips_list):,}")
        
        idx = 0
        max_batches = total_batches if total_batches > 0 else len(trips_list) // self.batch_size
        
        while self.running and idx < len(trips_list) and self.batch_count < max_batches:
            chunk = trips_list[idx:idx + self.batch_size]
            if not chunk:
                break
            
            batch_data, events = self.emit_batch_historical(chunk)
            filepath = self.write_batch(batch_data)
            
            pressure = batch_data['pressure_metrics']
            
            if self.batch_count % 10 == 0 or self.batch_count <= 3:
                logger.info(
                    f"📤 Batch {self.batch_count:4d} | "
                    f"Events: {len(events):3d} | "
                    f"Pressure: {pressure['pressure_index']:.2f} | "
                    f"Total: {self.total_events:,}"
                )
            
            idx += self.batch_size
            time.sleep(self.interval_sec)
        
        self.running = False
        logger.info(f"\n🏁 Simulação histórica encerrada!")
        logger.info(f"   Batches: {self.batch_count} | Events: {self.total_events:,}")
    
    def stop(self):
        """Para a simulação."""
        self.running = False
        logger.info("⏹️  Simulador parado")

print("✅ Classe CitiBikeEventSimulator definida")

✅ Classe CitiBikeEventSimulator definida


---
## 3. 🔥 Spark Structured Streaming (Consumidor)

Configuramos o Spark Structured Streaming para:
1. Monitorar o diretório `streaming/events/` por novos micro-lotes
2. Processar cada lote assim que ele chega
3. Calcular agregações em janelas temporais (windowed aggregations)
4. Gravar a saída processada

In [8]:
# ============================================================
# 3.1 Definir schema do stream e iniciar o consumidor
# ============================================================

# Schema dos eventos JSON
EVENT_SCHEMA = StructType([
    StructField('ride_id', StringType(), True),
    StructField('rideable_type', StringType(), True),
    StructField('started_at', StringType(), True),
    StructField('ended_at', StringType(), True),
    StructField('start_station_name', StringType(), True),
    StructField('start_station_id', StringType(), True),
    StructField('end_station_name', StringType(), True),
    StructField('end_station_id', StringType(), True),
    StructField('start_lat', DoubleType(), True),
    StructField('start_lng', DoubleType(), True),
    StructField('end_lat', DoubleType(), True),
    StructField('end_lng', DoubleType(), True),
    StructField('member_casual', StringType(), True),
    StructField('trip_duration_sec', LongType(), True),
])

# Criar o stream reader
stream_df = (
    spark.readStream
    .format('json')
    .schema(EVENT_SCHEMA)
    .option('maxFilesPerTrigger', 5)  # processar até 5 arquivos por trigger
    .load(str(STREAM_EVENTS))
)

print(f"✅ Stream reader configurado")
print(f"   Monitorando: {STREAM_EVENTS}")
print(f"   Schema: {EVENT_SCHEMA.fieldNames()}")

✅ Stream reader configurado
   Monitorando: /home/joao/citibike/streaming/events
   Schema: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'trip_duration_sec']


In [9]:
# ============================================================
# 3.2 Transformações no stream — Agregações por estação
# ============================================================

# Enriquecer o stream com timestamp parsed
enriched_stream = (
    stream_df
    .withColumn('event_timestamp', F.to_timestamp('started_at'))
    .withColumn('hour_of_day', F.hour('event_timestamp'))
    .withColumn('processing_time', F.current_timestamp())
)

# Agregação: contagem de partidas por estação (movimentação)
departures_agg = (
    enriched_stream
    .groupBy('start_station_id', 'start_station_name')
    .agg(
        F.count('*').alias('total_departures'),
        F.avg('trip_duration_sec').alias('avg_duration_sec'),
        F.approx_count_distinct('end_station_id').alias('unique_destinations'),
    )
)

# Agregação: contagem de chegadas por estação
arrivals_agg = (
    enriched_stream
    .groupBy('end_station_id', 'end_station_name')
    .agg(
        F.count('*').alias('total_arrivals'),
    )
)

print("✅ Transformações de stream definidas")

✅ Transformações de stream definidas


In [10]:
for q in spark.streams.active:
    q.stop()
    print(f"⏹️ Query '{q.name}' parada")

In [11]:
# ============================================================
# 3.3 Iniciar o stream writer (saída para memória — consulta interativa)
# ============================================================

# Query 1: Partidas por estação (atualizado a cada micro-lote)
query_departures = (
    departures_agg
    .writeStream
    .outputMode('complete')
    .format('memory')
    .queryName('station_departures')
    .trigger(processingTime='5 seconds')
    .start()
)

# Query 2: Chegadas por estação
query_arrivals = (
    arrivals_agg
    .writeStream
    .outputMode('complete')
    .format('memory')
    .queryName('station_arrivals')
    .trigger(processingTime='5 seconds')
    .start()
)

# Query 3: Todos os eventos (append) — salva em Parquet
query_all_events = (
    enriched_stream
    .writeStream
    .outputMode('append')
    .format('parquet')
    .option('path', str(STREAM_OUTPUT / 'events_parquet'))
    .option('checkpointLocation', str(STREAM_CHECKPOINT / 'events'))
    .trigger(processingTime='5 seconds')
    .start()
)

print("✅ Streaming queries iniciadas:")
print(f"   1. station_departures (memory sink)")
print(f"   2. station_arrivals (memory sink)")
print(f"   3. events_parquet (parquet sink → {STREAM_OUTPUT / 'events_parquet'})")

26/03/29 23:44:12 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-abfc532e-2468-42cd-b97d-20f9a16cd05a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/29 23:44:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/29 23:44:12 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


✅ Streaming queries iniciadas:
   1. station_departures (memory sink)
   2. station_arrivals (memory sink)
   3. events_parquet (parquet sink → /home/joao/citibike/streaming/output/events_parquet)


26/03/29 23:44:12 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-ebf8e990-fc22-4906-990c-637bf6299d37. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/29 23:44:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/29 23:44:12 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/03/29 23:44:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


---
## 4. 🚀 Executar Simulação

Agora vamos rodar o simulador e o consumidor Spark Streaming simultaneamente.

In [12]:
# ============================================================
# 4.1 Iniciar o simulador em uma thread separada
# ============================================================

simulator = CitiBikeEventSimulator(
    output_dir=STREAM_EVENTS,
    batch_size=BATCH_SIZE,
    interval_sec=BATCH_INTERVAL_SEC,
    speed_factor=SPEED_FACTOR,
)

# Escolher modo baseado na disponibilidade de dados
if total_trips > 0:
    # Modo histórico — usa dados reais da Bronze
    sim_thread = threading.Thread(
        target=simulator.run_historical,
        args=(day_df, TOTAL_BATCHES),
        daemon=True
    )
    mode = 'HISTÓRICO'
else:
    # Modo sintético — gera dados realistas
    sim_thread = threading.Thread(
        target=simulator.run_synthetic,
        args=(TOTAL_BATCHES, SIMULATION_DATE),
        daemon=True
    )
    mode = 'SINTÉTICO'

print(f"🚀 Iniciando simulador em modo {mode}...")
print(f"   O simulador vai rodar em background enquanto o Spark consome.")
print(f"   Execute a próxima célula para monitorar o progresso.")

sim_thread.start()

23:44:12 [INFO] 📖 Convertendo dados históricos para iteração...


🚀 Iniciando simulador em modo HISTÓRICO...
   O simulador vai rodar em background enquanto o Spark consome.
   Execute a próxima célula para monitorar o progresso.


In [13]:
# ============================================================
# 4.2 Monitorar o stream em tempo real
# ============================================================

# Aguardar alguns batches serem processados
import time

print("⏳ Aguardando batches serem processados...")
time.sleep(15)  # esperar 15s para acumular dados

# Consultar resultados em tempo real
print("\n" + "=" * 70)
print("📊 DASHBOARD — Partidas por Estação (top 10)")
print("=" * 70)
spark.sql("""
    SELECT 
        start_station_name,
        total_departures,
        ROUND(avg_duration_sec, 0) as avg_duration,
        unique_destinations
    FROM station_departures
    ORDER BY total_departures DESC
    LIMIT 10
""").show(truncate=False)

print("\n" + "=" * 70)
print("📊 DASHBOARD — Chegadas por Estação (top 10)")
print("=" * 70)
spark.sql("""
    SELECT 
        end_station_name,
        total_arrivals
    FROM station_arrivals
    ORDER BY total_arrivals DESC
    LIMIT 10
""").show(truncate=False)

# Status do simulador
print(f"\n📡 Status do simulador:")
print(f"   Batches emitidos: {simulator.batch_count}")
print(f"   Eventos totais: {simulator.total_events:,}")
print(f"   Rodando: {simulator.running}")
print(f"   Arquivos no stream dir: {len(list(STREAM_EVENTS.glob('*.json')))}")

⏳ Aguardando batches serem processados...


23:44:24 [INFO] 🚀 Simulador iniciado (modo HISTÓRICO)                          
23:44:24 [INFO]    Viagens disponíveis: 146,831
23:44:24 [INFO] 📤 Batch    1 | Events:  50 | Pressure: 0.00 | Total: 50
26/03/29 23:44:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
23:44:26 [INFO] 📤 Batch    2 | Events:  50 | Pressure: 0.00 | Total: 100
23:44:28 [INFO] 📤 Batch    3 | Events:  50 | Pressure: 0.00 | Total: 150



📊 DASHBOARD — Partidas por Estação (top 10)
+-------------------------+----------------+------------+-------------------+
|start_station_name       |total_departures|avg_duration|unique_destinations|
+-------------------------+----------------+------------+-------------------+
|W 50 St & 10 Ave         |2               |1438.0      |2                  |
|W 22 St & 8 Ave          |2               |1537.0      |1                  |
|Market St & Cherry St    |2               |176.0       |2                  |
|12 Ave & W 40 St         |1               |1794.0      |1                  |
|3 Ave & Franklin Ave     |1               |276.0       |1                  |
|W 54 St & 9 Ave          |1               |915.0       |1                  |
|31 St & Northern Blvd    |1               |96.0        |1                  |
|Rutgers St & Henry St    |1               |681.0       |1                  |
|W 170 St & University Ave|1               |7925.0      |1                  |
|Crown St & Troy Av

In [14]:
# ============================================================
# 4.3 Análise de pressão logística (Desbalanceamento)
# ============================================================

# Calcular net flow por estação (partidas - chegadas)
print("\n" + "=" * 70)
print("🔴 PRESSÃO LOGÍSTICA — Desbalanceamento por Estação")
print("=" * 70)

try:
    net_flow = spark.sql("""
        SELECT 
            COALESCE(d.start_station_name, a.end_station_name) as station,
            COALESCE(d.total_departures, 0) as departures,
            COALESCE(a.total_arrivals, 0) as arrivals,
            COALESCE(a.total_arrivals, 0) - COALESCE(d.total_departures, 0) as net_flow,
            CASE 
                WHEN COALESCE(a.total_arrivals, 0) - COALESCE(d.total_departures, 0) > 3 THEN '🔴 ACÚMULO'
                WHEN COALESCE(a.total_arrivals, 0) - COALESCE(d.total_departures, 0) < -3 THEN '🟡 DEFICIT'
                ELSE '🟢 OK'
            END as status
        FROM station_departures d
        FULL OUTER JOIN station_arrivals a 
            ON d.start_station_id = a.end_station_id
        ORDER BY net_flow ASC
    """)
    net_flow.show(20, truncate=False)
except Exception as e:
    print(f"⚠️  Aguardando mais dados: {e}")


🔴 PRESSÃO LOGÍSTICA — Desbalanceamento por Estação
+---------------------------+----------+--------+--------+------+
|station                    |departures|arrivals|net_flow|status|
+---------------------------+----------+--------+--------+------+
|W 22 St & 8 Ave            |2         |0       |-2      |🟢 OK |
|W 50 St & 10 Ave           |2         |0       |-2      |🟢 OK |
|Driggs Ave & S 2 St        |1         |0       |-1      |🟢 OK |
|Maple St & Flatbush Ave    |1         |0       |-1      |🟢 OK |
|Prospect Park West & 8 St  |1         |0       |-1      |🟢 OK |
|Crown St & Troy Ave        |1         |0       |-1      |🟢 OK |
|Franklin Ave & Quincy St   |1         |0       |-1      |🟢 OK |
|Nevins St & Schermerhorn St|1         |0       |-1      |🟢 OK |
|Wilson Ave & Troutman St   |1         |0       |-1      |🟢 OK |
|Hart St & Wyckoff Ave      |1         |0       |-1      |🟢 OK |
|Albany St & Greenwich St   |1         |0       |-1      |🟢 OK |
|Market St & Cherry St      |2     

In [15]:
# ============================================================
# 4.4 Aguardar conclusão e parar tudo
# ============================================================

# Aguardar o simulador terminar
sim_thread.join(timeout=300)  # timeout de 5 min

# Aguardar o Spark processar os últimos batches
time.sleep(10)

# Parar streaming queries
for q in spark.streams.active:
    q.stop()
    print(f"⏹️  Query '{q.name}' parada")

print(f"\n🏁 Simulação completa!")
print(f"   Total de micro-lotes: {simulator.batch_count}")
print(f"   Total de eventos: {simulator.total_events:,}")

23:44:42 [INFO] 📤 Batch   10 | Events:  50 | Pressure: 0.00 | Total: 500
23:45:05 [INFO] 📤 Batch   20 | Events:  50 | Pressure: 0.00 | Total: 1,000
23:45:25 [INFO] 📤 Batch   30 | Events:  50 | Pressure: 0.00 | Total: 1,500
23:45:48 [INFO] 📤 Batch   40 | Events:  50 | Pressure: 0.00 | Total: 2,000
23:46:10 [INFO] 📤 Batch   50 | Events:  50 | Pressure: 0.00 | Total: 2,500
23:46:30 [INFO] 📤 Batch   60 | Events:  50 | Pressure: 0.00 | Total: 3,000
23:46:53 [INFO] 📤 Batch   70 | Events:  50 | Pressure: 0.01 | Total: 3,500
23:47:15 [INFO] 📤 Batch   80 | Events:  50 | Pressure: 0.01 | Total: 4,000
23:47:35 [INFO] 📤 Batch   90 | Events:  50 | Pressure: 0.01 | Total: 4,500
23:47:58 [INFO] 📤 Batch  100 | Events:  50 | Pressure: 0.01 | Total: 5,000
23:48:00 [INFO] 
🏁 Simulação histórica encerrada!
23:48:00 [INFO]    Batches: 100 | Events: 5,000
23:48:00 [INFO] Closing down clientserver connection


⏹️  Query 'station_departures' parada
⏹️  Query 'station_arrivals' parada
⏹️  Query 'None' parada

🏁 Simulação completa!
   Total de micro-lotes: 100
   Total de eventos: 5,000


26/03/29 23:48:10 WARN DAGScheduler: Failed to cancel job group 43ef2711-c536-4858-a9d8-515148c72981. Cannot find active jobs for it.
26/03/29 23:48:10 WARN DAGScheduler: Failed to cancel job group 43ef2711-c536-4858-a9d8-515148c72981. Cannot find active jobs for it.
26/03/29 23:48:10 WARN DAGScheduler: Failed to cancel job group 98be85ec-28ce-4a95-b14d-59adaf0dd2bc. Cannot find active jobs for it.
26/03/29 23:48:10 WARN DAGScheduler: Failed to cancel job group 98be85ec-28ce-4a95-b14d-59adaf0dd2bc. Cannot find active jobs for it.
26/03/29 23:48:10 WARN DAGScheduler: Failed to cancel job group 1d792c2f-59fd-49fd-bca5-7353946ea24f. Cannot find active jobs for it.
26/03/29 23:48:10 WARN DAGScheduler: Failed to cancel job group 1d792c2f-59fd-49fd-bca5-7353946ea24f. Cannot find active jobs for it.


---
## 5. 📋 Validação & Resumo

In [16]:
# ============================================================
# 5.1 Verificar dados gravados pelo streaming
# ============================================================

output_parquet = STREAM_OUTPUT / 'events_parquet'
if output_parquet.exists() and any(output_parquet.rglob('*.parquet')):
    processed = spark.read.parquet(str(output_parquet))
    print(f"📊 Eventos processados pelo Streaming:")
    print(f"   Total de registros: {processed.count():,}")
    print(f"   Colunas: {processed.columns}")
    processed.show(5, truncate=False)
    
    size = sum(f.stat().st_size for f in output_parquet.rglob('*.parquet'))
    print(f"\n💾 Tamanho Parquet output: {size / 1e6:.1f} MB")
else:
    print("⚠️  Nenhum Parquet de streaming encontrado")

# Contar JSONs gerados
json_files = list(STREAM_EVENTS.glob('*.json'))
json_size = sum(f.stat().st_size for f in json_files)
print(f"\n📄 Micro-lotes JSON gerados: {len(json_files)}")
print(f"💾 Tamanho total JSONs: {json_size / 1e6:.1f} MB")

📊 Eventos processados pelo Streaming:
   Total de registros: 5,000
   Colunas: ['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'trip_duration_sec', 'event_timestamp', 'hour_of_day', 'processing_time']
+----------------+-------------+--------------------------+--------------------------+--------------------------+----------------+-----------------------------+--------------+-----------+------------+------------------+------------------+-------------+-----------------+-----------------------+-----------+-----------------------+
|ride_id         |rideable_type|started_at                |ended_at                  |start_station_name        |start_station_id|end_station_name             |end_station_id|start_lat  |start_lng   |end_lat           |end_lng           |member_casual|trip_duration_sec|event_timestamp        |hour_of_day|processing

In [17]:
# ============================================================
# 5.2 Balanço final das estações
# ============================================================

print("\n" + "=" * 70)
print("📊 BALANÇO FINAL DAS ESTAÇÕES (após simulação)")
print("=" * 70)

balance_data = []
for station in POPULAR_STATIONS:
    sid = station['id']
    bal = simulator.station_balance[sid]
    balance_data.append({
        'station': station['name'],
        'capacity': station['capacity'],
        'bikes': bal['bikes'],
        'docks': bal['docks'],
        'occupancy': f"{bal['bikes']/station['capacity']*100:.0f}%",
        'status': '🔴' if bal['bikes'] <= 2 else ('🟡' if bal['bikes'] <= 5 else '🟢')
    })

balance_df = pd.DataFrame(balance_data)
display(balance_df)


📊 BALANÇO FINAL DAS ESTAÇÕES (após simulação)


,station,capacity,bikes,docks,occupancy,status
0,Broadway & W 60 St,55,33,22,60%,🟢
1,Lincoln Center,39,23,16,59%,🟢
2,University Pl & E 14 St,51,30,21,59%,🟢
3,Grand Army Plaza & Central Park S,63,37,26,59%,🟢
4,W 21 St & 6 Ave,43,25,18,58%,🟢
5,E 53 St & Madison Ave,35,21,14,60%,🟢
6,Central Park W & W 72 St,47,28,19,60%,🟢
7,Bergen St & Smith St,25,15,10,60%,🟢
8,Fulton St & Broadway,41,7,34,17%,🟢
9,Pier 40 - Hudson River Park,59,35,24,59%,🟢


In [18]:
# ============================================================
# 5.3 Checklist final
# ============================================================

print("\n" + "=" * 70)
print("✅ CHECKLIST — ETAPA DE STREAMING (AV1)")
print("=" * 70)

checks = {
    'Simulador de eventos implementado': True,
    'Micro-lotes JSON gerados': len(json_files) > 0,
    'Spark Structured Streaming consumidor': True,
    'Agregações por estação (partidas/chegadas)': True,
    'Métricas de pressão logística': True,
    'Output em Parquet': output_parquet.exists() if output_parquet else False,
    'Perfil de demanda por hora do dia': True,
    'Balanço de bikes por estação': True,
}

for item, status in checks.items():
    icon = '✅' if status else '❌'
    print(f"  {icon} {item}")

print(f"\n  Progresso: {sum(checks.values())}/{len(checks)}")


✅ CHECKLIST — ETAPA DE STREAMING (AV1)
  ✅ Simulador de eventos implementado
  ✅ Micro-lotes JSON gerados
  ✅ Spark Structured Streaming consumidor
  ✅ Agregações por estação (partidas/chegadas)
  ✅ Métricas de pressão logística
  ✅ Output em Parquet
  ✅ Perfil de demanda por hora do dia
  ✅ Balanço de bikes por estação

  Progresso: 8/8


In [19]:
# Encerrar
spark.stop()
print("🏁 SparkSession encerrada. Simulador concluído!")

🏁 SparkSession encerrada. Simulador concluído!
